# load the csv 
# and get the shape ,type 
# descriptive stats = mean , median of each column
# check if there are missing values
# the first 5 rows

In [53]:
# starter code

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


df = pd.read_csv("lab_ticket_resolution.csv")
print(df.shape) # it tells u number of cols and rows
print(df.dtypes)
print(df.describe(include="all"))
print(df.isna().sum())  # there are no null values

(240, 6)
ticket_id                    str
complexity_score           int64
lines_changed              int64
team_experience_years    float64
reopened                   int64
resolution_hours         float64
dtype: object
       ticket_id  complexity_score  lines_changed  team_experience_years  \
count        240        240.000000     240.000000             240.000000   
unique       240               NaN            NaN                    NaN   
top        T0001               NaN            NaN                    NaN   
freq           1               NaN            NaN                    NaN   
mean         NaN          5.491667      54.820833               6.105417   
std          NaN          2.979697      26.145017               3.391426   
min          NaN          1.000000       1.000000               0.500000   
25%          NaN          3.000000      33.000000               3.300000   
50%          NaN          5.500000      54.000000               5.800000   
75%          NaN   

# define resolution_hours - y ie the value to be predicted
# exclude ticket_id bc we dont need it
# to prevent data leakage we only want the available things at the time of the prediction to be there
# these are - complexity_score , team_experience_years 
# lines_changed , reopened are available for an existing project not a new one 
# training data should only have the things we will have at the time of prediction.
# since this is a supervised learning we will provide the label in the training data.

In [54]:
training_data_prep = df.drop(columns=["lines_changed" , "reopened" , "ticket_id"])

# print(training_data_prep.shape)
# all rows are there but the columns are reduced

# spliting the data into training and test sets
# training data gets randomly shuffled when we choose so
# we have to protect testing data and training data from mixing up.
# spliting steps
# split label and feature - using []
# split training_feature, testing_feature ,training_label , testing_label = train_test( feature , label) 

In [55]:
label_set = training_data_prep["resolution_hours"]
feature_set = training_data_prep[["team_experience_years" , "complexity_score"]]



feature_to_train_on , feature_to_test_on , label_to_train_on ,label_to_test_on  = train_test_split(
    feature_set , label_set , test_size=0.2 , random_state=42
)

# Dummy Regressor always predicts the value you give it
# here we gave it mean thus it will always give the mean as a predicted value
# used for testing your model , your model needs to be smarter than this prediction
# Test metrics are judges on how well our model is fitting the training sets

In [56]:
# Dummy Regressor
dummy_model = DummyRegressor(strategy="mean")
# All predictions will be equal the the mean of the label data

dummy_model.fit(feature_to_train_on , label_to_train_on)
# make the model make the prediction ie get the mean of the label of training data
# so it can always use that

# for referene
# print(feature_to_train_on.describe(include="all")) mean - 6.25 and 5.45
# print(label_to_train_on.describe(include="all")) mean - 19.057031

# the line the dummy will produce is y = mean_of_trainings_label
# lets make the model predict on our testing set
# given the features this is the model's predicted labels
dummy_predictions  = dummy_model.predict(feature_to_test_on)
# print(dummy_predictions)


# testing if the data model fits the training data well is checked by checking the errors
# mse , mae , r_2 .. are used to check the deviation between predicted and real values
# u can use them both in training and testing data
# if u use them on testing data it will be better check for validity

# print(dummy_predictions.shape)
# print(label_to_test_on.shape)

# mae - the average of the deviation between actual and predicted values
mae = mean_absolute_error(label_to_test_on ,dummy_predictions)

# # mse - rss / n - squared sum of residuals / n
mse = mean_squared_error(label_to_test_on  , dummy_predictions)



# # the better test , variablity_captured_by_the_line / variablity of the population
# # if r2 is great it means our model is good
r2 = r2_score(label_to_test_on ,dummy_predictions )



print("mae = " , mae )
print("mse = " , mse)
print("r squared = " , r2 )
# r2 is negative which implies it is a bad model
# the model isn't even trying to predict on the test dataset



mae =  9.113951822916667
mse =  112.31092313639323
r squared =  -0.021016765260001957


# trying to fit the data better using linear regression models
# the linear regression tries to create a linear line to fit the pattern it sees in the training data

In [ ]:
linear_model = LinearRegression()

linear_model.fit(feature_to_train_on , label_to_train_on)

linear_predictions  = linear_model.predict(feature_to_test_on)


# mae - the average of the deviation between actual and predicted values
linear_mae = mean_absolute_error(label_to_test_on ,linear_predictions)

# # mse - rss / n - squared sum of residuals / n
linear_mse = mean_squared_error(label_to_test_on  , linear_predictions)



# # the better test , variablity_captured_by_the_line / variablity of the population
# # if r2 is great it means our model is good
linear_r2 = r2_score(label_to_test_on , linear_predictions )



print("mae = " , linear_mae )
print("mse = " , linear_mse)
print("r squared = " , linear_r2 )
# the mae , mse ,  are far lower here than the dummy so linear is better than dummy
# r squared is higher - 0.884 so it means the linear model covered much of the pattern and distribution seen in the testing set



mae =  2.90601298595026
mse =  12.761144066531239
r squared =  0.8839886480141969
